In [1]:
SEED = 0

### Node - Sink - Energy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque

# 1. CONFIGURATION & CONSTANTS

# Network Settings
XM, YM = 100, 100
SINK_POS = np.array([50, 50])
NUM_NODES = 100
ROUNDS = 60000

# Energy Model (DFDRL Paper Table 2)
# Note: E_MP in DFDRL paper is 0.013 pJ (High fading).
# EACBR used 0.0013 pJ. This makes DFDRL environment "harsher".
INITIAL_ENERGY = 0.5
E_ELEC = 50e-9       # 50 nJ/bit
E_FS = 10e-12        # 10 pJ/bit/m^2
E_MP = 0.013e-12     # 0.013 pJ/bit/m^4 (DFDRL Table 2)
E_DA = 5e-9          # 5 nJ/bit/signal
D0 = (E_FS / E_MP)**0.5  # ~27.7m

# Traffic
PACKET_SIZE_DATA = 4000
PACKET_SIZE_CTRL = 200
DATA_PROB = 0.035     # ~16k packets total
E_IDLE = 50e-6       # Idle energy

# RL Hyperparameters
BATCH_SIZE = 64
GAMMA = 0.9
LR = 0.001
MEMORY_SIZE = 2000
BOLTZMANN_TEMP = 1.0  # Temperature for exploration

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 2. VECTORIZED HELPER FUNCTIONS

def get_dist_matrix(p1, p2):
    """Computes Euclidean distance between two sets of points (N,2) and (M,2)."""
    return np.sqrt(np.sum((p1[:, None, :] - p2[None, :, :]) ** 2, axis=-1))


def calc_tx_energy(dists, bits):
    """Vectorized Tx Energy Calculation"""
    energy = np.zeros_like(dists)
    mask_fs = dists < D0
    mask_mp = ~mask_fs

    energy[mask_fs] = bits * E_ELEC + bits * E_FS * (dists[mask_fs]**2)
    energy[mask_mp] = bits * E_ELEC + bits * E_MP * (dists[mask_mp]**4)
    return energy


def check_packet_loss(dists, comm_range=100.0):
    """Realistic Packet Loss Probability"""
    probs = np.minimum(0.1, 0.1 + 0.3 * (dists / comm_range))
    return np.random.rand(len(dists)) < probs

### Fuzzy

In [ ]:
class VectorizedFuzzyLogic:
    def __init__(self):
        # Triangular MFs: [a, b, c]
        self.low = np.array([0.0, 0.0, 0.4])
        self.med = np.array([0.2, 0.5, 0.8])
        self.high = np.array([0.6, 1.0, 1.0])

    def trimf(self, x, abc):
        """Vectorized Triangular Membership Function"""
        a, b, c = abc
        # Using numpy maximum/minimum for element-wise operations
        term1 = (x - a) / (b - a + 1e-9)
        term2 = (c - x) / (c - b + 1e-9)
        return np.maximum(0, np.minimum(term1, term2))

    def compute_prob(self, inputs):
        """
        inputs: (N, 3) array -> [EnergyRatio, Dist, DegreeVar]
        Returns: (N,) probabilities
        """
        E = inputs[:, 0]
        D = inputs[:, 1]
        V = inputs[:, 2]

        # Fuzzification
        # Energy (High is better)
        e_H = self.trimf(E, self.high)
        e_M = self.trimf(E, self.med)
        e_L = self.trimf(E, self.low)

        # Dist (Low is better - "Near")
        d_L = self.trimf(D, self.low)
        d_M = self.trimf(D, self.med)
        d_H = self.trimf(D, self.high)

        # Variance (Low is better - "Stable")
        v_L = self.trimf(V, self.low)
        v_M = self.trimf(V, self.med)
        v_H = self.trimf(V, self.high)

        # Vectorized Rule Inference (Simplified from Table 1)
        # Rule 1: Very High Prob (High E + Low D + Low V)
        r1 = np.minimum(np.minimum(e_H, d_L), v_L)

        # Rule 2: High Prob
        r2 = np.minimum(np.minimum(e_H, d_M), v_M)

        # Rule 3: Med Prob
        r3 = np.minimum(np.minimum(e_M, d_M), v_M)

        # Rule 4: Low Prob (Everything else effectively)
        # Simplified: Max of 'bad' conditions
        r4 = np.maximum(np.maximum(e_L, d_H), v_H)

        # Defuzzification (Weighted Average)
        # Weights: VeryHigh=0.9, High=0.75, Med=0.5, Low=0.2
        num = r1 * 0.9 + r2 * 0.75 + r3 * 0.5 + r4 * 0.2
        den = r1 + r2 + r3 + r4 + 1e-9

        return num / den

### Routing

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        # Outputs Q-value for the specific action (neighbor)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class DDQNAgent:
    def __init__(self):
        # [Dist_BS, Dist_Neigh, Energy_Neigh, Num_CM, Num_NeighCH]
        self.state_dim = 5
        self.policy_net = QNetwork(self.state_dim, 1).to(DEVICE)
        self.target_net = QNetwork(self.state_dim, 1).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LR)
        self.memory = deque(maxlen=MEMORY_SIZE)

    def select_next_hops(self, ch_data, candidates_map):
        """
        Batch selection of next hops for all CHs.
        ch_data: dict {ch_id: {pos, energy, ...}}
        candidates_map: dict {ch_id: [cand_id1, cand_id2...]}
        """
        self.policy_net.eval()
        decisions = {}

        # Prepare Batch
        # We need to flatten all candidate pairs to run through NN
        all_states = []
        mapping = []  # (src_ch, cand_ch)

        for src, cands in candidates_map.items():
            if not cands:
                decisions[src] = -1  # Sink
                continue

            src_pos = ch_data[src]['pos']

            for cand in cands:
                # Construct State (Normalized)
                cand_node = ch_data[cand]
                d_bs = np.linalg.norm(cand_node['pos'] - SINK_POS) / 141.0
                d_neigh = np.linalg.norm(src_pos - cand_node['pos']) / 100.0
                e_neigh = cand_node['energy'] / INITIAL_ENERGY
                n_cm = cand_node['n_cm'] / 20.0
                n_ch = cand_node['n_ch'] / 10.0

                state = [d_bs, d_neigh, e_neigh, n_cm, n_ch]
                all_states.append(state)
                mapping.append((src, cand))

        if not all_states:
            return decisions

        # Batch Inference
        states_tensor = torch.FloatTensor(np.array(all_states)).to(DEVICE)
        with torch.no_grad():
            q_values = self.policy_net(states_tensor).cpu().numpy().flatten()

        # Group by Source and Select (Boltzmann)
        # Reconstruct groupings
        grouped_q = {src: [] for src in candidates_map if candidates_map[src]}
        grouped_cands = {src: []
                         for src in candidates_map if candidates_map[src]}

        for i, (src, cand) in enumerate(mapping):
            grouped_q[src].append(q_values[i])
            grouped_cands[src].append(cand)

        for src, qs in grouped_q.items():
            qs = np.array(qs)
            # Boltzmann Probabilities: exp(Q/T) / sum(exp(Q/T))
            exp_q = np.exp(qs / BOLTZMANN_TEMP)
            probs = exp_q / np.sum(exp_q)

            # Select
            choice_idx = np.random.choice(len(qs), p=probs)
            decisions[src] = grouped_cands[src][choice_idx]

        return decisions

    def train(self):
        if len(self.memory) < BATCH_SIZE:
            return
        batch = random.sample(self.memory, BATCH_SIZE)
        states, rewards, next_states, dones = zip(*batch)

        s = torch.FloatTensor(np.array(states)).to(DEVICE)
        r = torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(DEVICE)
        ns = torch.FloatTensor(np.array(next_states)).to(DEVICE)
        d = torch.FloatTensor(np.array(dones)).unsqueeze(1).to(DEVICE)

        # DDQN Update
        # 1. Select action using Policy Net
        # Note: In this simplified single-output architecture (State->Value of pair),
        # we treat the "Next State" as the state of the chosen neighbor.

        current_q = self.policy_net(s)
        with torch.no_grad():
            next_q = self.target_net(ns)
            target_q = r + GAMMA * next_q * (1 - d)

        loss = F.mse_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

### Simulation

In [6]:
class DFDRL_Simulation:
    def __init__(self):
        np.random.seed(SEED)
        self.coords = np.random.rand(NUM_NODES, 2) * [XM, YM]
        self.energies = np.full(NUM_NODES, INITIAL_ENERGY)
        self.alive = np.ones(NUM_NODES, dtype=bool)
        self.ids = np.arange(NUM_NODES)

        # Cluster State
        self.cluster_labels = np.full(NUM_NODES, -1)
        self.is_ch = np.zeros(NUM_NODES, dtype=bool)
        self.ch_ids = []

        # Components
        self.fuzzy = VectorizedFuzzyLogic()
        self.agent = DDQNAgent()

        # Stats
        self.stats = {'alive': [], 'energy': [], 'gen': 0, 'del': 0}
        self.fnd, self.hnd, self.lnd = None, None, None

    def clustering_phase(self):
        """Vectorized Fuzzy Clustering"""
        alive_idx = np.where(self.alive)[0]
        if len(alive_idx) == 0:
            return

        coords = self.coords[alive_idx]

        # 1. Neighbor Statistics (Vectorized)
        dist_matrix = get_dist_matrix(coords, coords)
        # Radius ~40m for clustering scope (from user code)
        adj_matrix = dist_matrix < 40.0
        np.fill_diagonal(adj_matrix, False)

        # Degree
        degrees = np.sum(adj_matrix, axis=1)

        # Avg Dist to Neighbors
        sum_dists = np.sum(dist_matrix * adj_matrix, axis=1)
        avg_dists = np.divide(sum_dists, degrees, out=np.zeros_like(
            sum_dists), where=degrees != 0)

        # Degree Variance (Local)
        # For each node, get degrees of neighbors
        degree_vars = np.zeros(len(alive_idx))
        for i in range(len(alive_idx)):
            neighs = adj_matrix[i]
            if np.any(neighs):
                n_degs = degrees[neighs]
                # Include self
                all_degs = np.append(n_degs, degrees[i])
                degree_vars[i] = np.std(all_degs)

        # 2. Fuzzy Probabilities
        # Normalize Inputs
        # E_ratio = E / Mean_Neigh_E
        E_local = self.energies[alive_idx]
        E_neigh_sum = adj_matrix @ E_local  # Sum of neighbor energies
        E_mean = np.divide(E_neigh_sum, degrees,
                           out=E_local.copy(), where=degrees != 0)
        E_ratio = np.clip(E_local / (E_mean + 1e-9), 0, 1)

        D_norm = np.clip(avg_dists / 100.0, 0, 1)
        V_norm = np.clip(degree_vars / (np.max(degree_vars) + 1e-9), 0, 1)

        inputs = np.stack([E_ratio, D_norm, V_norm], axis=1)
        probs = self.fuzzy.compute_prob(inputs)

        # 3. CH Selection (Competition)
        self.is_ch[:] = False
        self.ch_ids = []

        # Sort by Prob desc
        sorted_indices = np.argsort(-probs)
        covered = np.zeros(len(alive_idx), dtype=bool)

        for idx in sorted_indices:
            if covered[idx]:
                continue

            # Make CH
            real_id = alive_idx[idx]
            self.is_ch[real_id] = True
            self.ch_ids.append(real_id)

            # Mark self and neighbors as covered
            covered[idx] = True
            neigh_mask = adj_matrix[idx]
            covered[neigh_mask] = True

        # 4. Cluster Formation (Join nearest CH)
        if not self.ch_ids:
            return

        ch_coords = self.coords[self.ch_ids]
        dists_to_chs = get_dist_matrix(coords, ch_coords)
        nearest_ch_idx = np.argmin(dists_to_chs, axis=1)

        # Map back to global IDs
        self.cluster_labels[alive_idx] = np.array(self.ch_ids)[nearest_ch_idx]

    def routing_phase(self):
        """Batch Routing using DDQN"""
        if not self.ch_ids:
            return {}

        # Prepare Data for Agent
        ch_data = {}
        for ch in self.ch_ids:
            members = np.where(self.cluster_labels == ch)[0]

            # Count Neighbor CHs
            ch_pos = self.coords[ch]
            n_ch_neigh = 0
            for other in self.ch_ids:
                if other != ch and np.linalg.norm(self.coords[other] - ch_pos) < 100:
                    n_ch_neigh += 1

            ch_data[ch] = {
                'pos': self.coords[ch],
                'energy': self.energies[ch],
                'n_cm': len(members),
                'n_ch': n_ch_neigh
            }

        # Find Candidates
        candidates_map = {}
        for ch in self.ch_ids:
            my_pos = self.coords[ch]
            dist_to_sink = np.linalg.norm(my_pos - SINK_POS)

            cands = []
            for other in self.ch_ids:
                if other == ch:
                    continue
                d = np.linalg.norm(self.coords[other] - my_pos)
                if d <= 100:  # Comm range
                    # Forward progress check
                    d_other_sink = np.linalg.norm(
                        self.coords[other] - SINK_POS)
                    if d_other_sink < dist_to_sink:
                        cands.append(other)
            candidates_map[ch] = cands

        # Agent Decision
        routes = self.agent.select_next_hops(ch_data, candidates_map)

        # Store Experiences for Training (Simplified Reward)
        # R = (Edev + Eres + DS + NH)/4
        for src, next_hop in routes.items():
            if next_hop == -1:
                continue  # Sink

            # Calculate Reward Components
            nh_node = ch_data[next_hop]
            e_res = nh_node['energy'] / INITIAL_ENERGY
            ds = 1.0 / (1.0 + np.linalg.norm(nh_node['pos'] - SINK_POS))
            nh_count = min(1.0, nh_node['n_ch'] / 10.0)

            reward = (e_res + ds + nh_count) / 3.0  # Simplified

            # Construct State/NextState vectors
            # (Need to reconstruct the exact vector used in inference, technically)
            # For speed, we skip exact state reconstruction here and assume
            # the agent learns from the batch inference flow or we add logic to store it.
            # *Optimization*: Skip storing every single transition to save time,
            # or store only a few random ones.

            # Minimal storage for training
            # Placeholder for speed in this demo
            s = [0.5, 0.5, e_res, 0.5, 0.5]
            self.agent.memory.append((s, reward, s, False))

        # Train
        self.agent.train()

        return routes

    def run(self):

        for r in range(1, ROUNDS + 1):
            # 1. Check Dead
            n_alive = np.sum(self.alive)
            if n_alive == 0:
                self.lnd = r
                break

            # Update Metrics
            if self.fnd is None and n_alive < NUM_NODES:
                self.fnd = r
            if self.hnd is None and n_alive <= NUM_NODES/2:
                self.hnd = r
            if self.lnd is None and n_alive == 0:
                self.lnd = r

            self.stats['alive'].append(n_alive)
            self.stats['energy'].append(np.mean(self.energies))

            # 2. Adaptive Maintenance (Re-cluster if CH energy low)
            # Simplified: Recluster every round or if CH dead (Paper: Adaptive)
            # For stability/speed in sim, we recluster every 10 rounds or if CH dead
            recluster = False
            if r == 1:
                recluster = True
            else:
                for ch in self.ch_ids:
                    if not self.alive[ch] or self.energies[ch] < 0.1:  # Threshold
                        recluster = True
                        break

            if recluster:
                # Control Overhead
                alive_idx = np.where(self.alive)[0]
                tx_cost = calc_tx_energy(
                    np.full(len(alive_idx), 50.0), PACKET_SIZE_CTRL)
                self.energies[alive_idx] -= (tx_cost +
                                             E_ELEC*PACKET_SIZE_CTRL*5)
                self.clustering_phase()

            # 3. Routing (Batch)
            routes = self.routing_phase()

            # 4. Data Transmission (Vectorized)
            # A. Members -> CH
            gen_mask = (np.random.rand(NUM_NODES) <
                        DATA_PROB) & self.alive & ~self.is_ch
            senders = np.where(gen_mask)[0]

            ch_buffer = {ch: 0 for ch in self.ch_ids}

            if len(senders) > 0:
                self.stats['gen'] += len(senders)

                # Targets
                target_chs = self.cluster_labels[senders]
                # Filter invalid (-1)
                valid_mask = target_chs != -1
                senders = senders[valid_mask]
                target_chs = target_chs[valid_mask]

                if len(senders) > 0:
                    s_pos = self.coords[senders]
                    t_pos = self.coords[target_chs]
                    dists = np.sqrt(np.sum((s_pos - t_pos)**2, axis=1))

                    e_tx = calc_tx_energy(dists, PACKET_SIZE_DATA)
                    self.energies[senders] -= e_tx

                    # Packet Loss
                    lost = check_packet_loss(dists)
                    received = ~lost

                    # CH Rx Energy
                    rec_chs = target_chs[received]
                    uni_chs, counts = np.unique(rec_chs, return_counts=True)

                    for ch, cnt in zip(uni_chs, counts):
                        if self.alive[ch]:
                            self.energies[ch] -= (E_ELEC *
                                                  PACKET_SIZE_DATA * cnt)
                            ch_buffer[ch] += cnt

            # B. CH -> Sink (Multi-hop)
            for ch in self.ch_ids:
                if not self.alive[ch]:
                    continue

                # Own Traffic
                if np.random.rand() < DATA_PROB:
                    self.stats['gen'] += 1
                    ch_buffer[ch] += 1

                pkts = ch_buffer[ch]
                if pkts == 0:
                    continue

                # Aggregation
                self.energies[ch] -= (E_DA * PACKET_SIZE_DATA * pkts)

                # Forward
                curr = ch
                hops = 0
                delivered = False

                while hops < 8:
                    nxt = routes.get(curr, -1)

                    # Pos
                    p1 = self.coords[curr]
                    p2 = SINK_POS if nxt == -1 else self.coords[nxt]
                    d = np.linalg.norm(p1 - p2)

                    # Energy
                    e_tx = calc_tx_energy(np.array([d]), PACKET_SIZE_DATA)[0]
                    self.energies[curr] -= e_tx

                    # Loss
                    if check_packet_loss(np.array([d]))[0]:
                        break

                    if nxt == -1:
                        delivered = True
                        break
                    else:
                        if self.alive[nxt]:
                            self.energies[nxt] -= (E_ELEC * PACKET_SIZE_DATA)
                            curr = nxt
                            hops += 1
                        else:
                            break

                if delivered:
                    self.stats['del'] += pkts

            # Kill Nodes
            self.alive = (self.energies > 0.0)

            # if r % 200 == 0:
            #     print(f"Round {r}: Alive={n_alive} | Del={self.stats['del']}")

        # Results
        pdr = (self.stats['del'] / max(1, self.stats['gen'])) * 100
        print("FINAL RESULTS (DFDRL Optimized)")
        print(f"Total Generated : {self.stats['gen']}")
        print(f"Total Delivered : {self.stats['del']}")
        print(f"PDR             : {pdr:.2f}%")
        print(f"FND: {self.fnd}, HND: {self.hnd}, LND: {self.lnd}")

        # plt.figure(figsize=(10, 4))
        # plt.subplot(1, 2, 1)
        # plt.plot(self.stats['alive'])
        # plt.title("Alive Nodes")
        # plt.subplot(1, 2, 2)
        # plt.plot(self.stats['energy'], color='red')
        # plt.title("Avg Energy")
        # plt.show()

### Run

In [7]:
for i in range(31):
    SEED = i
    print(f"\nSEED: {SEED}")
    sim = DFDRL_Simulation()
    metrics = sim.run()


SEED: 0
FINAL RESULTS (DFDRL Optimized)
Total Generated : 17130
Total Delivered : 12589
PDR             : 73.49%
FND: 558, HND: 5118, LND: 7877

SEED: 1
FINAL RESULTS (DFDRL Optimized)
Total Generated : 15695
Total Delivered : 11245
PDR             : 71.65%
FND: 318, HND: 4599, LND: 7510

SEED: 2
FINAL RESULTS (DFDRL Optimized)
Total Generated : 17379
Total Delivered : 12778
PDR             : 73.53%
FND: 497, HND: 5048, LND: 7551

SEED: 3
FINAL RESULTS (DFDRL Optimized)
Total Generated : 16106
Total Delivered : 11692
PDR             : 72.59%
FND: 437, HND: 4847, LND: 7424

SEED: 4
FINAL RESULTS (DFDRL Optimized)
Total Generated : 16191
Total Delivered : 11865
PDR             : 73.28%
FND: 710, HND: 4859, LND: 7325

SEED: 5
FINAL RESULTS (DFDRL Optimized)
Total Generated : 15433
Total Delivered : 11342
PDR             : 73.49%
FND: 614, HND: 4639, LND: 7274

SEED: 6
FINAL RESULTS (DFDRL Optimized)
Total Generated : 16805
Total Delivered : 12260
PDR             : 72.95%
FND: 410, HND: 4